# Lecture 6: Merkle Transparency Logs

Transparency logs make signed statements auditable. PACTA uses an RFC 9162-style Merkle accumulator over signed proof-check attestations. A provider signs the tree head, and an agent verifies an inclusion proof before acting on the attestation.


## Learning Objectives

- Implement leaf and node hashing with domain separation.
- Compute a Merkle root.
- Generate and verify inclusion proofs.
- Generate and verify consistency proofs.
- Explain Signed Tree Heads and signature policy.
- Explain why ML-DSA must fail closed when unavailable.


## RFC 9162 Hash Shape

PACTA follows the Certificate Transparency hash structure:

- Empty tree hash: `SHA256("")`
- Leaf hash: `SHA256(0x00 || leaf_input)`
- Node hash: `SHA256(0x01 || left || right)`

The prefix bytes prevent a leaf value from being confused with an internal node value.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.transparency import (
    leaf_hash,
    node_hash,
    merkle_root,
    inclusion_proof,
    verify_inclusion,
    consistency_proof,
    verify_consistency,
)

leaves = [f"attestation-{i}".encode() for i in range(1, 6)]
root = merkle_root(leaves)
print(root.hex())


In [ ]:
for index, leaf in enumerate(leaves):
    proof = inclusion_proof(leaves, index)
    ok = verify_inclusion(leaf, index, len(leaves), proof, root)
    print(index, ok, [node.hex()[:12] for node in proof])


## Consistency Proofs

An inclusion proof answers: "Is this leaf in this tree?"

A consistency proof answers: "Is the newer tree an append-only extension of the older tree?"

Both are needed for a monitored transparency system. Inclusion is enough for one agent to bind one attestation to one signed tree head. Consistency lets monitors detect equivocation or tree rewrites across time.


In [ ]:
old_size = 3
old_root = merkle_root(leaves[:old_size])
new_root = merkle_root(leaves)
proof = consistency_proof(leaves, old_size)
print("old:", old_root.hex())
print("new:", new_root.hex())
print("proof:", [node.hex()[:12] for node in proof])
print("consistent:", verify_consistency(old_size, len(leaves), old_root, new_root, proof))


## Signed Tree Heads

A Signed Tree Head records:

- log ID,
- tree size,
- timestamp,
- root hash,
- hash algorithm,
- signatures.

PACTA signs the canonical JSON STH payload with Ed25519 through OpenSSL. It also records an ML-DSA-65 slot. On this host, if no real ML-DSA backend is present, the slot is `unavailable`.

Policy matters:

- `require-signatures ed25519`: verify Ed25519 and allow ML-DSA to be unavailable.
- `require-signatures both`: require Ed25519 and ML-DSA verified. If ML-DSA is unavailable, fail closed.


In [ ]:
from pacta.postquantum import detect_ml_dsa

capability = detect_ml_dsa()
print(capability.available)
print(capability.backend)
print(capability.reason)


## Why Ed25519 and ML-DSA Together?

Ed25519 is useful because it is widely deployed, fast, and directly relevant to the Ed25519 proof corpus. That creates a deliberate "eat your own dogfood" loop: the proof-checking ecosystem signs evidence using a primitive whose implementation family is under formal scrutiny.

ML-DSA adds post-quantum robustness for the accumulator signature layer. But it must be a real signature, not an aspirational label. If a host lacks ML-DSA, the correct result is an explicit blocker.


## Exercises

- Tamper with one leaf and show that inclusion verification fails.
- Explain why the tree head signature must cover tree size as well as root hash.
- Write a policy for when an autonomous agent should require `both` signatures.
- Research checkpoint: compare PACTA's local prototype to production Certificate Transparency monitor/gossip requirements.
